In [1]:
from kaggle_secrets import UserSecretsClient
secret_label = "Token"
token = UserSecretsClient().get_secret(secret_label)

! git clone https://{token}@github.com/PietroGhersetich/RecSys_challenge_2025

Cloning into 'RecSys_challenge_2025'...
remote: Enumerating objects: 1576, done.
remote: Counting objects: 100% (217/217), done.
remote: Compressing objects: 100% (216/216), done.
remote: Total 1576 (delta 145), reused 0 (delta 0), pack-reused 1359 (from 1)
Receiving objects: 100% (1576/1576), 627.33 MiB | 51.92 MiB/s, done.
Resolving deltas: 100% (973/973), done.
Updating files: 100% (224/224), done.


In [2]:
! pip install PyGithub requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.6 MB/s eta 0:00:00


In [3]:
! cd /kaggle/working/RecSys_challenge_2025 && python run_compile_all_cython.py

run_compile_all_cython: Found 11 Cython files in 5 folders...
run_compile_all_cython: All files will be compiled using your current python environment: '/usr/bin/python3'
Compiling [1/11]: MatrixFactorizationImpressions_Cython_Epoch.pyx... 
In file included from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/ndarraytypes.h:1929,
                 from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/ndarrayobject.h:12,
                 from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/arrayobject.h:5,
                 from MatrixFactorizationImpressions_Cython_Epoch.c:1252:
/usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wcpp-Wcpp]8;;]
   17 | #warning "Using deprecated NumPy API, disable it with " \
 

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import random
import tqdm
import gc
import time
import os
import shutil
import optuna
from optuna.distributions import FloatDistribution, IntDistribution, CategoricalDistribution
import scipy.sparse as sps

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv
/kaggle/input/recommender-systems-2025-challenge-polimi/sample_submission.csv
/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv


In [5]:
from github import Github
from github import Auth

auth = Auth.Token(token)
g = Github(auth=auth)

repo = None
for r in g.get_user().get_repos():
    if r.name == 'RecSys_challenge_2025':
        repo = r
        print('Repository found')

Repository found


In [6]:
def upload_file(filepath_kaggle, filepath_github, commit_message):
    try:
        contents = repo.get_contents(filepath_github)
        with open(filepath_kaggle, "rb") as file:
            repo.update_file(contents.path, commit_message, file.read(), contents.sha)
    except Exception:
        with open(filepath_kaggle, "rb") as file:
            repo.create_file(filepath_github, commit_message, file.read())

In [7]:
db_name = 'ials_study.db'
json_name = 'ials_best_params.json'

config = {
    'tune_parameters': True,
    'database_name': db_name,
    'database_path': f'/kaggle/working/{db_name}',
    'save_json': True,
    'json_name': json_name,
    'copy_prev_best_params': False,
    'save_github': True,
    'candidates': '50',
    'n_trials': 20
}

In [8]:
try:
    shutil.copyfile(
        f'/kaggle/working/RecSys_challenge_2025/tuning_results/recall@{config["candidates"]}/{config["database_name"]}', 
        config['database_path']
    )
except FileNotFoundError:
    pass # if not present optuna will create it

# Data Import

In [9]:
train_data = pd.read_csv('/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv')
user_ids_test = pd.read_csv('/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv')

In [10]:
data = np.ones(len(train_data))

URM_all = sps.csr_matrix((data, (train_data['row'].values, train_data['col'].values)),
                         shape=(train_data['row'].max() + 1, train_data['col'].max() + 1))

URM_all

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3043058 stored elements and shape (27095, 6969)>

In [11]:
%cd /kaggle/working/RecSys_challenge_2025

/kaggle/working/RecSys_challenge_2025


In [12]:
from Recommenders.BaseRecommender import BaseRecommender
from Recommenders.MatrixFactorization.IALSRecommender import IALSRecommender
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample

from Evaluation.Evaluator import EvaluatorHoldout

np.int = int
np.random.seed(42)
np.bool = bool
np.float = float

In [13]:
URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_all, train_percentage = 0.8)

In [14]:
import sqlite3

def print_all_rows(table_name, database_path):
    connection = sqlite3.connect(database_path)
    cursor = connection.cursor()

    try:
        cursor.execute(f"SELECT * FROM {table_name};")
        rows = cursor.fetchall()

        print(f"All rows from table '{table_name}':")
        for row in rows:
            print(row)

    except sqlite3.Error as e:
        print(f"Error reading data from table '{table_name}': {e}")

    finally:
        cursor.close()
        connection.close()

print_all_rows('studies', config['database_path'])

All rows from table 'studies':
(1, 'item_ials_Optimization')


In [15]:
evaluator = EvaluatorHoldout(URM_validation, cutoff_list=[int(config['candidates'])])

def objective_function_ials(trial):
    params = {
        'num_factors': trial.suggest_int('num_factors', 5, 300), 
        'reg': trial.suggest_float('reg', 1e-5, 100, log=True),
        'alpha': trial.suggest_float('alpha', 1e-3, 50.0, log=True),
        'confidence_scaling': trial.suggest_categorical('confidence_scaling', ['linear', 'log'])
    }
    
    if params['confidence_scaling'] == 'log':
        params['epsilon'] = trial.suggest_float('epsilon', 1e-3, 10.0, log=True)

    evaluator_validation = EvaluatorHoldout(URM_validation, cutoff_list=[int(config['candidates'])])
    early_stopping_params = {
        'epochs_min': 5,
        'stop_on_validation': True,
        'validation_every_n': 5,
        'validation_metric': 'RECALL',
        'lower_validations_allowed': 5,
        'evaluator_object': evaluator_validation
    }

    recommender = IALSRecommender(URM_train)
    recommender.fit(**params, **early_stopping_params)
    
    epochs = recommender.get_early_stopping_final_epochs_dict()["epochs"]
    trial.set_user_attr("epochs", epochs) 

    result_df, _ = evaluator.evaluateRecommender(recommender)
    
    return result_df.loc[int(config['candidates']), 'RECALL']

EvaluatorHoldout: Ignoring 28 ( 0.1%) Users that have less than 1 test interactions


In [16]:
%%time

if config['tune_parameters']:
    study = optuna.create_study(direction='maximize', study_name='item_ials_Optimization', 
                                storage=f'sqlite:///{config["database_path"]}', load_if_exists=True)
    print(f"Previous trials found: {len(study.trials)}")
    study.optimize(objective_function_ials, n_trials=config['n_trials'])

[I 2025-12-30 22:54:26,105] Using an existing study with name 'item_ials_Optimization' instead of creating a new one.


Previous trials found: 111
EvaluatorHoldout: Ignoring 28 ( 0.1%) Users that have less than 1 test interactions
IALSRecommender: Epoch 1 of 300. Elapsed time 8.39 sec
IALSRecommender: Epoch 2 of 300. Elapsed time 16.70 sec
IALSRecommender: Epoch 3 of 300. Elapsed time 24.97 sec
IALSRecommender: Epoch 4 of 300. Elapsed time 33.34 sec
IALSRecommender: Validation begins...
EvaluatorHoldout: Processed 27067 (100.0%) in 17.54 sec. Users per second: 1543
IALSRecommender: CUTOFF: 50 - PRECISION: 0.1350730, PRECISION_RECALL_MIN_DEN: 0.3989404, RECALL: 0.3898646, MAP: 0.0476407, MAP_MIN_DEN: 0.1289152, MRR: 0.4867610, NDCG: 0.3149475, F1: 0.2006340, HIT_RATE: 0.9769461, ARHR_ALL_HITS: 0.9402787, NOVELTY: 0.0712748, AVERAGE_POPULARITY: 0.2870055, DIVERSITY_MEAN_INTER_LIST: 0.8833283, DIVERSITY_HERFINDAHL: 0.9976659, COVERAGE_ITEM: 0.2861243, COVERAGE_ITEM_HIT: 0.2386282, ITEMS_IN_GT: 0.9991390, COVERAGE_USER: 0.9989666, COVERAGE_USER_HIT: 0.9759365, USERS_IN_GT: 0.9989666, DIVERSITY_GINI: 0.06638

[I 2025-12-30 23:18:16,271] Trial 111 finished with value: 0.3992380157518134 and parameters: {'num_factors': 71, 'reg': 73.28724988970126, 'alpha': 0.7890885149479099, 'confidence_scaling': 'log', 'epsilon': 0.006366282801598753}. Best is trial 91 with value: 0.4109542025703591.


EvaluatorHoldout: Ignoring 28 ( 0.1%) Users that have less than 1 test interactions
IALSRecommender: Epoch 1 of 300. Elapsed time 11.09 sec
IALSRecommender: Epoch 2 of 300. Elapsed time 22.21 sec
IALSRecommender: Epoch 3 of 300. Elapsed time 33.29 sec
IALSRecommender: Epoch 4 of 300. Elapsed time 44.51 sec
IALSRecommender: Validation begins...
EvaluatorHoldout: Processed 27067 (100.0%) in 17.37 sec. Users per second: 1558
IALSRecommender: CUTOFF: 50 - PRECISION: 0.1307880, PRECISION_RECALL_MIN_DEN: 0.4009430, RECALL: 0.3929886, MAP: 0.0425660, MAP_MIN_DEN: 0.1228315, MRR: 0.4575396, NDCG: 0.3071870, F1: 0.1962600, HIT_RATE: 0.9783500, ARHR_ALL_HITS: 0.8712281, NOVELTY: 0.0727573, AVERAGE_POPULARITY: 0.2663330, DIVERSITY_MEAN_INTER_LIST: 0.9039207, DIVERSITY_HERFINDAHL: 0.9980777, COVERAGE_ITEM: 0.4159851, COVERAGE_ITEM_HIT: 0.3429473, ITEMS_IN_GT: 0.9991390, COVERAGE_USER: 0.9989666, COVERAGE_USER_HIT: 0.9773390, USERS_IN_GT: 0.9989666, DIVERSITY_GINI: 0.0917136, SHANNON_ENTROPY: 9.716

[I 2025-12-30 23:31:59,876] Trial 112 finished with value: 0.40509069819395965 and parameters: {'num_factors': 90, 'reg': 43.81150798543782, 'alpha': 2.7344150997533614, 'confidence_scaling': 'log', 'epsilon': 0.00458240276614605}. Best is trial 91 with value: 0.4109542025703591.


...

EvaluatorHoldout: Ignoring 28 ( 0.1%) Users that have less than 1 test interactions
IALSRecommender: Epoch 1 of 300. Elapsed time 13.10 sec
IALSRecommender: Epoch 2 of 300. Elapsed time 26.30 sec
IALSRecommender: Epoch 3 of 300. Elapsed time 39.47 sec
IALSRecommender: Epoch 4 of 300. Elapsed time 52.73 sec
IALSRecommender: Validation begins...
EvaluatorHoldout: Processed 27067 (100.0%) in 18.95 sec. Users per second: 1428
IALSRecommender: CUTOFF: 50 - PRECISION: 0.1299390, PRECISION_RECALL_MIN_DEN: 0.3913180, RECALL: 0.3831484, MAP: 0.0431715, MAP_MIN_DEN: 0.1227115, MRR: 0.4612795, NDCG: 0.3048208, F1: 0.1940641, HIT_RATE: 0.9754313, ARHR_ALL_HITS: 0.8801762, NOVELTY: 0.0734271, AVERAGE_POPULARITY: 0.2474565, DIVERSITY_MEAN_INTER_LIST: 0.9185612, DIVERSITY_HERFINDAHL: 0.9983705, COVERAGE_ITEM: 0.3970441, COVERAGE_ITEM_HIT: 0.3340508, ITEMS_IN_GT: 0.9991390, COVERAGE_USER: 0.9989666, COVERAGE_USER_HIT: 0.9744233, USERS_IN_GT: 0.9989666, DIVERSITY_GINI: 0.0972747, SHANNON_ENTROPY: 9.827

[I 2025-12-31 06:41:36,334] Trial 129 finished with value: 0.39829793521687373 and parameters: {'num_factors': 98, 'reg': 0.0003362253049345128, 'alpha': 1.2713588302451961, 'confidence_scaling': 'log', 'epsilon': 0.0016348774753156933}. Best is trial 91 with value: 0.4109542025703591.


EvaluatorHoldout: Ignoring 28 ( 0.1%) Users that have less than 1 test interactions
IALSRecommender: Epoch 1 of 300. Elapsed time 14.00 sec
IALSRecommender: Epoch 2 of 300. Elapsed time 28.28 sec
IALSRecommender: Epoch 3 of 300. Elapsed time 42.26 sec
IALSRecommender: Epoch 4 of 300. Elapsed time 56.07 sec
IALSRecommender: Validation begins...
EvaluatorHoldout: Processed 27067 (100.0%) in 17.91 sec. Users per second: 1511
IALSRecommender: CUTOFF: 50 - PRECISION: 0.1325385, PRECISION_RECALL_MIN_DEN: 0.4028734, RECALL: 0.3947372, MAP: 0.0439815, MAP_MIN_DEN: 0.1261414, MRR: 0.4655464, NDCG: 0.3116192, F1: 0.1984460, HIT_RATE: 0.9777958, ARHR_ALL_HITS: 0.8918681, NOVELTY: 0.0730074, AVERAGE_POPULARITY: 0.2580435, DIVERSITY_MEAN_INTER_LIST: 0.9106403, DIVERSITY_HERFINDAHL: 0.9982121, COVERAGE_ITEM: 0.4029272, COVERAGE_ITEM_HIT: 0.3357727, ITEMS_IN_GT: 0.9991390, COVERAGE_USER: 0.9989666, COVERAGE_USER_HIT: 0.9767854, USERS_IN_GT: 0.9989666, DIVERSITY_GINI: 0.0930472, SHANNON_ENTROPY: 9.752

[I 2025-12-31 07:10:09,601] Trial 130 finished with value: 0.40861888157694615 and parameters: {'num_factors': 104, 'reg': 60.83331790265166, 'alpha': 1.993112030692507, 'confidence_scaling': 'log', 'epsilon': 0.0012489898675843022}. Best is trial 91 with value: 0.4109542025703591.


CPU times: user 14h 35min 7s, sys: 11min 22s, total: 14h 46min 30s
Wall time: 8h 15min 45s


In [17]:
if config['save_github'] and config['tune_parameters']:
    upload_file(
        config['database_path'], 
        f'tuning_results/recall@{config["candidates"]}/{config["database_name"]}',
        'Tuning db updated results (from kaggle notebook)'
    )

    if config['save_json']:
        j_name = config['json_name']
        l_path = f"/kaggle/working/{j_name}"
        r_path = f"tuning_results/recall@{config['candidates']}/best_params/{j_name}"
        
        best_val, best_params, best_trial_num, best_study_name, best_epochs = -float('inf'), {}, None, None, None
        
        try:
            for s in optuna.get_all_study_summaries(storage=f"sqlite:///{config['database_path']}"):
                if s.best_trial and s.best_trial.value > best_val:
                    best_val, best_params, best_trial_num, best_study_name, best_epochs = s.best_trial.value, s.best_trial.params, s.best_trial.number, s.study_name, s.best_trial.user_attrs.get("epochs")
        except:
            best_val, best_params, best_trial_num, best_study_name, best_epochs = study.best_value, study.best_params, study.best_trial.number, study.study_name, study.best_trial.user_attrs.get("epochs")

        if best_epochs is not None:
            best_params['epochs'] = int(best_epochs)
        
        prev_val, exists = -float('inf'), False
        try:
            prev_val = json.loads(repo.get_contents(r_path).decoded_content.decode()).get('value', -float('inf'))
            exists = True
        except:
            pass
    
        if not exists or best_val > prev_val:
            print(f"Updating JSON. Best: {best_val:.6f} (Prev: {prev_val:.6f})")
            with open(l_path, 'w') as f:
                json.dump({"value": best_val, "study_name": best_study_name, "trial_number": best_trial_num, "params": best_params}, f, indent=4)
            if config.get('save_github'):
                upload_file(l_path, r_path, f'Update params {j_name} (from kaggle notebook)')
        else:
            print(f"No update needed. Best: {best_val:.6f} <= Saved: {prev_val:.6f}")

No update needed. Best: 0.410954 <= Saved: 0.410954
